# ULTRON Training Control

Portable browser launcher for Kaggle, Google Colab, Lightning AI and GitHub Actions. It downloads the same training engine, deterministic mathematics generator, independent verifier, Transformer, checkpoint system and Supabase coordination code from this repository.

Run the validation cell first. The final cell starts a real training session and keeps running until it reaches a safe handoff, pause or completion. It resumes from the latest committed checkpoint and never includes a secret in the notebook or repository.

On Kaggle, enable Internet in Notebook Settings. Add `SUPABASE_SECRET_KEY` and `SUPABASE_OWNER_ID` to the platform's secret manager. Add `HF_TOKEN` and `HF_REPO_ID` to enable the private secondary checkpoint backup.

In [ ]:
import json
import pathlib
import subprocess
import sys
import urllib.parse
import urllib.request

REPOSITORY = "shayaanfarsheed-cmd/ULTRONAI"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{REPOSITORY}/{BRANCH}"
PROJECT_DIR = pathlib.Path.cwd() / ".ultron_training_control"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
FILES = ("train.py", "ultron_model.py", "ultron_data.py", "ultron_verify.py", "ultron_checkpoint.py", "ultron_training.py", "ultron_supabase.py", "ultron_storage.py", "ultron_session.py", "config.json", "servers.json", "requirements.txt")

for filename in FILES:
    request = urllib.request.Request(
        f"{BASE_URL}/{filename}",
        headers={"User-Agent": "ULTRON-Training-Control"},
    )
    with urllib.request.urlopen(request, timeout=45) as response:
        final_host = urllib.parse.urlparse(response.geturl()).hostname
        if final_host != "raw.githubusercontent.com":
            raise RuntimeError("Unexpected download host; refusing to save the project file.")
        content = response.read(2_000_001)
    if len(content) > 2_000_000:
        raise RuntimeError(f"Project file is unexpectedly large: {filename}")
    if filename.endswith(".json"):
        json.loads(content.decode("utf-8"))
    (PROJECT_DIR / filename).write_bytes(content)

try:
    import numpy
    import safetensors
    import huggingface_hub
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
        check=True,
    )

print("ULTRON project files retrieved from GitHub.")
print("Project folder:", PROJECT_DIR)

In [ ]:
command = [
    sys.executable,
    str(PROJECT_DIR / "train.py"),
    "--preflight",
    "--config",
    str(PROJECT_DIR / "config.json"),
    "--servers",
    str(PROJECT_DIR / "servers.json"),
]
result = subprocess.run(command, cwd=PROJECT_DIR, check=False)
if result.returncode != 0:
    raise RuntimeError(f"ULTRON preflight failed with exit code {result.returncode}")

summary_command = command + ["--model-summary"]
summary_result = subprocess.run(summary_command, cwd=PROJECT_DIR, check=False)
if summary_result.returncode != 0:
    raise RuntimeError(f"ULTRON model summary failed with exit code {summary_result.returncode}")

data_command = command + ["--data-smoke-test"]
data_result = subprocess.run(data_command, cwd=PROJECT_DIR, check=False)
if data_result.returncode != 0:
    raise RuntimeError(f"ULTRON data verification failed with exit code {data_result.returncode}")

checkpoint_command = command + ["--checkpoint-smoke-test"]
checkpoint_result = subprocess.run(checkpoint_command, cwd=PROJECT_DIR, check=False)
if checkpoint_result.returncode != 0:
    raise RuntimeError(f"ULTRON checkpoint smoke test failed with exit code {checkpoint_result.returncode}")

training_command = command + ["--training-smoke-test"]
training_result = subprocess.run(training_command, cwd=PROJECT_DIR, check=False)
if training_result.returncode != 0:
    raise RuntimeError(f"ULTRON training smoke test failed with exit code {training_result.returncode}")

## Secure Supabase connection

This cell reads credentials from Kaggle Secrets, Colab Secrets or the platform environment. It never prints or writes the key.

In [ ]:
import os
import uuid

def _read_platform_secret(name):
    existing = os.environ.get(name)
    if existing:
        return existing
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception:
        pass
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return None

config = json.loads((PROJECT_DIR / "config.json").read_text(encoding="utf-8"))
os.environ["SUPABASE_URL"] = config["database"]["project_url"]

missing = []
for secret_name in ("SUPABASE_SECRET_KEY", "SUPABASE_OWNER_ID"):
    secret_value = _read_platform_secret(secret_name)
    if not secret_value:
        missing.append(secret_name)
    else:
        os.environ[secret_name] = secret_value

if missing:
    raise RuntimeError("Add these values to this platform's secret manager before training: " + ", ".join(missing))

if not os.environ["SUPABASE_SECRET_KEY"].startswith("sb_secret_"):
    raise RuntimeError("Use a current Supabase secret key beginning with sb_secret_. Do not paste it into this notebook.")
try:
    uuid.UUID(os.environ["SUPABASE_OWNER_ID"])
except ValueError:
    raise RuntimeError("SUPABASE_OWNER_ID must be your Supabase Auth user UUID.")

hf_token = _read_platform_secret("HF_TOKEN")
hf_repo_id = _read_platform_secret("HF_REPO_ID")
if bool(hf_token) != bool(hf_repo_id):
    raise RuntimeError("Add both HF_TOKEN and HF_REPO_ID, or leave both unset.")
if hf_token and hf_repo_id:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HF_REPO_ID"] = hf_repo_id
    print("Private secondary checkpoint backup is configured. Secret values are hidden.")
else:
    print("Secondary checkpoint backup is not configured. Supabase is the only remote checkpoint store.")
print("Supabase credentials loaded from platform secret storage. Secret values are hidden.")


## Start or resume ULTRON training

Run this cell after validation and secret setup. It trains until a safe handoff, pause or completion.

In [ ]:
training_command = [
    sys.executable,
    str(PROJECT_DIR / "train.py"),
    "--train",
    "--config",
    str(PROJECT_DIR / "config.json"),
    "--servers",
    str(PROJECT_DIR / "servers.json"),
]
training_result = subprocess.run(training_command, cwd=PROJECT_DIR, check=False)
if training_result.returncode != 0:
    raise RuntimeError(f"ULTRON training session stopped with exit code {training_result.returncode}")
